In [1]:
import pandas as pd
from glob import glob
import re
import matplotlib.pyplot as plt
import json
import numpy as np
from scipy import stats as st
import ee
import shapely.geometry
from shapely.geometry import Point, Polygon
import geopandas as gpd
from math import sqrt
from shapely import wkt
import os
import time
import geemap

In [20]:
ee.Authenticate()
ee.Initialize(project='ext-datasets')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
year = 2017

In [6]:
# Choose appropriate ACZ

# agroclimatic_zone = "Eastern Plateau & Hills Region"
agroclimatic_zone = "Southern Plateau and Hills Region"
# agroclimatic_zone = "East Coast Plains & Hills Region"
# agroclimatic_zone = "Western Plateau and Hills Region"
# agroclimatic_zone = "Central Plateau & Hills Region"
# agroclimatic_zone = "Lower Gangetic Plain Region"
# agroclimatic_zone = "Middle Gangetic Plain Region"
# agroclimatic_zone = "Eastern Himalayan Region"
#agroclimatic_zone = "Western Himalayan Region"
# agroclimatic_zone = "Upper Gangetic Plain Region"
# agroclimatic_zone = "Trans Gangetic Plain Region"
# agroclimatic_zone = "West Coast Plains & Ghat Region" ## # Model not available
# agroclimatic_zone = "Gujarat Plains & Hills Region" ## # Model not available
# agroclimatic_zone = "Western Dry Region" ## # Model not available

In [7]:
agroclimaticZone_acronym_dict = {'Eastern Plateau & Hills Region': 'EPAHR',
                               'Southern Plateau and Hills Region': 'SPAHR',
                               'East Coast Plains & Hills Region': 'ECPHR',
                               'Western Plateau and Hills Region': 'WPAHR',
                               'Central Plateau & Hills Region': 'CPAHR',
                               'Lower Gangetic Plain Region': 'LGPR',
                                'Middle Gangetic Plain Region': 'MGPR',
                                'Eastern Himalayan Region': 'EHR',
                                'Western Himalayan Region': 'WHR',
                                'Upper Gangetic Plain Region': 'UGPR',
                                'Trans Gangetic Plain Region': 'TGPR',
                                'West Coast Plains & Ghat Region': 'WCPGR',
                                'Gujarat Plains & Hills Region': 'GPHR',
                                'Western Dry Region': 'WDR'}

In [8]:
india_boundary = ee.FeatureCollection("projects/ee-mtpictd/assets/harsh/Agroclimatic_regions")
agrozone = india_boundary.filter(ee.Filter.eq('regionname', agroclimatic_zone)).geometry()
india_district_boundary = ee.FeatureCollection("projects/ee-indiasat/assets/india_district_boundaries")

In [9]:
s1_bands = ['VV', 'VH', 'angle']
s2_bands = ['B1','B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B10', 'B11','B12']

In [21]:
START_DATE = {year-2: {'kharif': str(year-2)+'-07-01', 'rabi': str(year-2)+'-11-01', 'zaid': str(year-1)+'-03-01'},
              year-1: {'kharif': str(year-1)+'-07-01', 'rabi': str(year-1)+'-11-01', 'zaid': str(year)+'-03-01'},
              year: {'kharif': str(year)+'-07-01', 'rabi': str(year)+'-11-01', 'zaid': str(year+1)+'-03-01'}}

END_DATE = {year-2: {'kharif': str(year-2)+'-10-31', 'rabi': str(year-1)+'-02-28', 'zaid': str(year-1)+'-06-30'},
            year-1: {'kharif': str(year-1)+'-10-31', 'rabi': str(year)+'-02-28', 'zaid': str(year)+'-06-30'},
            year: {'kharif': str(year)+'-10-31', 'rabi': str(year+1)+'-02-28', 'zaid': str(year+1)+'-06-30'}}

In [22]:
print(START_DATE)
print(END_DATE)

{2015: {'kharif': '2015-07-01', 'rabi': '2015-11-01', 'zaid': '2016-03-01'}, 2016: {'kharif': '2016-07-01', 'rabi': '2016-11-01', 'zaid': '2017-03-01'}, 2017: {'kharif': '2017-07-01', 'rabi': '2017-11-01', 'zaid': '2018-03-01'}}
{2015: {'kharif': '2015-10-31', 'rabi': '2016-02-28', 'zaid': '2016-06-30'}, 2016: {'kharif': '2016-10-31', 'rabi': '2017-02-28', 'zaid': '2017-06-30'}, 2017: {'kharif': '2017-10-31', 'rabi': '2018-02-28', 'zaid': '2018-06-30'}}


In [12]:
# Will take an AOI geometry as input and return the 10km x 10km grids in it as a list
def createGrids(aoi):
    proj = ee.Projection('EPSG:4326')
    gridSize = 10000
    grid = aoi.coveringGrid(proj, gridSize)
    features = grid.getInfo()['features']
    return features

In [13]:
def s2_mask(image):
  """
  Getting a cloud-free Sentinel-2 imagery.
  """
  quality_band = image.select('QA60')
  # Using the bit mask for clouds (bit 10) and cirrus clouds (bit 11) respectively.
  cloudmask = 1 << 10
  cirrusmask = 1 << 11
  # Both flags should be set to zero, indicating clear conditions of sky.
  mask = quality_band.bitwiseAnd(cloudmask).eq(0) and (quality_band.bitwiseAnd(cirrusmask).eq(0))
  return image.updateMask(mask)

def get_s2_image(aoi, start_date, end_date):
  # s2_bands_season = [band + '_med_' + season for band in s2_bands]
  return ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filterDate(
      start_date , end_date).filterBounds(aoi).filter(
          ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)).sort('CLOUD_COVER').map(
              s2_mask).select(s2_bands).median().divide(10000).clip(aoi)

def get_s1_image(aoi, start_date, end_date):
  # s1_bands_season = [band + '_' + season for band in s1_bands]
  return ee.ImageCollection('COPERNICUS/S1_GRD').filterDate(start_date , end_date).filterBounds(aoi).filter(
      ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')).filter(
          ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')).filter(
              ee.Filter.eq('instrumentMode', 'IW')).select(s1_bands).median().clip(aoi)

In [14]:
def save_data_csv(data_points, img_name, district, year):
  print("Saving data for", district, year)
  new_img_name = img_name.replace('&', 'and')
  new_img_name = new_img_name.replace('(', '')
  new_img_name = new_img_name.replace(')', '')
  task = ee.batch.Export.table.toDrive(
      collection = data_points,
      description = new_img_name,
      folder = f"{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{year}", # f"{agroclimaticZone_acronym_dict[agroclimatic_zone]}_{district}_{year}",
      fileNamePrefix = new_img_name,
      fileFormat = 'CSV'
      )
  task.start()
  print("Task Started",task.status())
  return task

In [23]:
df = pd.read_csv(f'drive/MyDrive/TreeHealth/Agroclimatic_regions/{agroclimatic_zone}.csv')
dist_list = ['Y.S.R.']#list(df['Name'])
print(len(dist_list))
print(dist_list)

1
['Y.S.R.']


In [24]:
years = [str(int(year)-2), str(int(year)-1), str(year)]
year_0 = years[0]
year_1 = years[1]
year_2 = years[2]
year_suffix = {year_0: year_0[-2:], year_1: year_1[-2:], year_2: year_2[-2:]}
print(years)

['2015', '2016', '2017']


In [17]:
def get_dw_tree_cover(aoi, start_date, end_date, scale = 25):
    tree_cover_dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate(start_date, end_date) \
                  .filterBounds(aoi).select('label').mode().clip(aoi)
    return tree_cover_dw.updateMask(tree_cover_dw.eq(1)).reproject(crs='EPSG:4326', scale=scale)

def get_is_tree_cover(aoi, curr_year, scale = 25):
    curr_year = int(curr_year)
    indiasat_asset = f"projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_{curr_year}_{curr_year+1}"
    lulc_image = ee.Image(indiasat_asset).select("predicted_label").clip(aoi)
    return lulc_image.updateMask(lulc_image.eq(6)).reproject(crs='EPSG:4326', scale=scale)

def get_tree_cover(aoi, curr_year, scale = 25):
    curr_year = int(curr_year)
    start_date = ee.Date(f'{curr_year}-07-01')
    end_date = ee.Date(f'{curr_year+1}-06-30')
    print("curr_year", curr_year)
    tree_cover_is = get_is_tree_cover(aoi, curr_year, scale) if curr_year > 2016 else None
    tree_cover_dw = get_dw_tree_cover(aoi, start_date, end_date, scale)
    if tree_cover_is:
      tree_cover = tree_cover_is.mask().Or(tree_cover_dw.mask())
    else:
      print("=================Only DW")
      tree_cover = tree_cover_dw.mask()
    # tree_cover = tree_cover_is.mask().Or(tree_cover_dw.mask()) if tree_cover_is else tree_cover_dw.mask()
    tree_cover = tree_cover.updateMask(tree_cover)
    return tree_cover.reproject(crs='EPSG:4326', scale=scale)

# Combined Grid

In [25]:
# Combined tiles and optimized
import sys
sys.setrecursionlimit(6000)
for curr_year in ['2017']:#years:

    total_time = 0

    dist_cnt = 0
    for district in dist_list:

        start_time = time.time()

        district_aoi = india_district_boundary.filter(ee.Filter.eq('Name', district)).geometry()
        district_aoi = district_aoi.intersection(agrozone)
        features = createGrids(district_aoi)

        print(f'Year {curr_year}, District {dist_cnt}: {district}, grids: {len(features)}')

        path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{curr_year}/'
        os.makedirs(path, exist_ok=True)

        # df = pd.DataFrame()
        # df['grid_num'] = list(range(len(features)))
        valid_grid_indices = []
        tree_points_list = []

        # Precompute tree cover for the whole district
        tree_cover_district = get_tree_cover(district_aoi, curr_year, scale=25)

        # Precompute S1 and S2 images for the whole district for each season
        start_date = START_DATE[int(curr_year)]
        end_date = END_DATE[int(curr_year)]
        s1_images = {}
        s2_images = {}
        for season in ['kharif', 'rabi', 'zaid']:
            try:
                s1_images[season] = get_s1_image(district_aoi, start_date[season], end_date[season]).updateMask(tree_cover_district)
            except Exception as exp:
                print(f"S1 Error occured: {season}", exp)
                s1_images[season] = None
            try:
                s2_images[season] = get_s2_image(district_aoi, start_date[season], end_date[season]).updateMask(tree_cover_district)
            except Exception as exp:
                print(f"S2 Error occured: {season}", exp)
                s2_images[season] = None

        # List to hold all sample points
        all_sample_points = []

        i = 0
        for feature in features:
            print(f'Grid {i}')

            coord = feature['geometry']['coordinates'][0]
            aoi = ee.Geometry.Polygon(coord)
            aoi = aoi.intersection(district_aoi)
            # Get the coordinates of the geometry
            coordinates = aoi.coordinates()
            # Check if coordinates list is empty (i.e. geometry is empty)
            is_empty = coordinates.length().eq(0)

            # print('Is geometry empty?', is_empty.getInfo())
            if not is_empty.getInfo():
              valid_grid_indices.append(i)  # Only add if valid
              img_name = district + "_" + str(i) + "_" + str(curr_year)

              # Clip tree cover to grid
              tree_cover = tree_cover_district.clip(aoi)

              # Compose image for all seasons using precomputed images
              image = None
              if s1_images['kharif'] is not None:
                  image = s1_images['kharif'].clip(aoi)
              if s2_images['kharif'] is not None and image is not None:
                  s2_data = s2_images['kharif'].clip(aoi)
                  image = image.addBands(s2_data).select(s1_bands + s2_bands)
                  image = image.rename([band + '_kharif' for band in s1_bands + s2_bands])

              for season in ['rabi', 'zaid']:
                  s1_data = s1_images[season]
                  s2_data = s2_images[season]
                  if s1_data is not None:
                      s1_data = s1_data.clip(aoi)
                  if s2_data is not None and s1_data is not None:
                      s2_data = s2_data.clip(aoi)
                      image_merged = s1_data.addBands(s2_data).select(s1_bands + s2_bands)
                      image_merged = image_merged.rename([band + '_' + season for band in s1_bands + s2_bands])
                      image = image.addBands(image_merged) if image is not None else image_merged

              # Sample points only if tree cover exists
              sample_tree_cover = tree_cover.sample(
                  region=aoi,
                  scale=25,
                  factor=1,
                  tileScale=10,
                  geometries=True
              )
              try:
                  tree_points = sample_tree_cover.size().getInfo()
              except:
                  tree_points = 0
              tree_points_list.append(tree_points)

              # if tree_points > 0 and image is not None:
              sample_points = image.sample(
                  region=aoi,
                  scale=25,
                  factor=1,
                  tileScale=10,
                  geometries=True
              )
              all_sample_points.append(sample_points)

              i += 1
        print(all_sample_points)
        # Merge all sample points into a single FeatureCollection
        if all_sample_points:
            merged_sample_points = all_sample_points[0]
            for sp in all_sample_points[1:]:
                merged_sample_points = merged_sample_points.merge(sp)

            # Export the merged FeatureCollection
            try:
              img_name = district + "_" + str(curr_year) + "_all_grids"
              task = save_data_csv(merged_sample_points, img_name, district, curr_year)
              prev_task = task
            except Exception as e:
                print(e)
                while prev_task.status()['state'] != 'COMPLETED' and prev_task.status()['state'] != 'FAILED':
                    continue
                task = save_data_csv(merged_sample_points, img_name, path, curr_year)
                prev_task = task

        df = pd.DataFrame()
        df['grid_num'] = valid_grid_indices
        df['tree_points'] = tree_points_list
        df.to_csv(path + 'tree_cover_points.csv', index=False)

        dist_cnt += 1
        end_time = time.time()

        total_time += (end_time - start_time)
        print(total_time)

    # print("Waiting for last task to be completed...")
    # while prev_task.status()['state'] != 'COMPLETED' and prev_task.status()['state'] != 'FAILED':
    #     continue
    # print("Last task completed!")

    # total_time += (time.time() - end_time)
    # print("Total Time Taken:", total_time)

Year 2017, District 0: Y.S.R., grids: 201
curr_year 2017
Grid 0
Grid 1
Grid 2
Grid 3
Grid 4
Grid 5
Grid 6
Grid 7
Grid 8
Grid 9
Grid 10
Grid 11
Grid 12
Grid 13
Grid 14
Grid 15
Grid 16
Grid 17
Grid 18
Grid 19
Grid 20
Grid 21
Grid 22
Grid 23
Grid 24
Grid 25
Grid 26
Grid 27
Grid 28
Grid 29
Grid 30
Grid 31
Grid 32
Grid 33
Grid 34
Grid 35
Grid 36
Grid 37
Grid 38
Grid 39
Grid 40
Grid 41
Grid 42
Grid 43
Grid 44
Grid 45
Grid 46
Grid 47
Grid 48
Grid 49
Grid 50
Grid 51
Grid 52
Grid 53
Grid 54
Grid 55
Grid 56
Grid 57
Grid 58
Grid 59
Grid 60
Grid 61
Grid 62
Grid 63
Grid 64
Grid 65
Grid 66
Grid 67
Grid 68
Grid 69
Grid 70
Grid 71
Grid 72
Grid 73
Grid 74
Grid 75
Grid 76
Grid 77
Grid 78
Grid 79
Grid 80
Grid 81
Grid 82
Grid 83
Grid 84
Grid 85
Grid 86
Grid 87
Grid 88
Grid 89
Grid 90
Grid 91
Grid 92
Grid 93
Grid 94
Grid 95
Grid 96
Grid 97
Grid 98
Grid 99
Grid 100
Grid 101
Grid 102
Grid 103
Grid 104
Grid 105
Grid 106
Grid 107
Grid 108
Grid 109
Grid 110
Grid 111
Grid 112
Grid 113
Grid 114
Grid 115
Grid 116


# Separate Grid (Older version - Do not run this)

In [ ]:
# years = ['2016']
for curr_year in years:

  total_time = 0

  dist_cnt = 0
  for district in dist_list:

      # if dist_cnt < 3:
      #     dist_cnt += 1
      #     continue

      start_time = time.time()

      district_aoi = india_district_boundary.filter(ee.Filter.eq('Name', district)).geometry()
      district_aoi = district_aoi.intersection(agrozone)
      features = createGrids(district_aoi)

      print(f'Year {curr_year}, District {dist_cnt}: {district}, grids: {len(features)}')

      path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/{district}/{curr_year}/'
      os.makedirs(path, exist_ok=True)

      df = pd.DataFrame()
      df['grid_num'] = list(range(len(features)))
      tree_points_list = []

      i = 0
      for feature in features:
          print(f'Grid {i}')

          coord = feature['geometry']['coordinates'][0]
          aoi = ee.Geometry.Polygon(coord)
          aoi = aoi.intersection(district_aoi)

          img_name = district + "_" + str(i) + "_" + str(curr_year)

          start_date = START_DATE[int(curr_year)]
          end_date = END_DATE[int(curr_year)]

          # Get tree cover using both Dynamic World and IndiaSat (union)
          tree_cover = get_tree_cover(aoi, curr_year, scale=25)

          season = 'kharif'
          try:
              image = get_s1_image(aoi, start_date[season], end_date[season])
              image = image.updateMask(tree_cover)
          except Exception as exp:
              print("S1 Error occured: ", season, exp)

          try:
              s2_data = get_s2_image(aoi, start_date[season], end_date[season])
              s2_data = s2_data.updateMask(tree_cover)
              image = image.addBands(s2_data).select(s1_bands + s2_bands)
              image = image.rename([band + '_kharif' for band in s1_bands + s2_bands])
          except Exception as exp:
              print("S2 Error occured: ", season, exp)

          for season in ['rabi', 'zaid']:
              try:
                  s1_data = get_s1_image(aoi, start_date[season], end_date[season])
                  s1_data = s1_data.updateMask(tree_cover)

              except Exception as exp:
                  print("S1 Error occured: ", season, exp)

              try:
                  s2_data = get_s2_image(aoi, start_date[season], end_date[season])
                  s2_data = s2_data.updateMask(tree_cover)
                  image_merged = s1_data.addBands(s2_data).select(s1_bands + s2_bands)
                  image_merged = image_merged.rename([band + '_' + season for band in s1_bands + s2_bands])
                  image = image.addBands(image_merged)

              except Exception as exp:
                  print("S2 Error occured: ", season, exp)

          sample_points = image.sample(
                  region = aoi,
                  scale = 25,
                  factor = 1,
                  tileScale = 10,
                  geometries = True
              )

          sample_tree_cover = tree_cover.sample(
              region = aoi,
              scale = 25,
              factor = 1,
              tileScale = 10,
              geometries = True
          )

          try:
              tree_points_list.append(sample_tree_cover.size().getInfo())
          except:
              tree_points_list.append(0)

          try:
              task = save_data_csv(sample_points, img_name, district, curr_year)
              prev_task = task
              # tasks[curr_year][district] = task

          except Exception as e:
              print(e)
              while prev_task.status()['state'] != 'COMPLETED' and prev_task.status()['state'] != 'FAILED':
                  # time.sleep(10)
                  continue
              task = save_data_csv(sample_points, img_name, path, curr_year)
              # tasks[curr_year][district] = task
              prev_task = task

          i += 1

      df['tree_points'] = tree_points_list
      df.to_csv(path + 'tree_cover_points.csv', index=False)

      dist_cnt += 1
      end_time = time.time()

      total_time += (end_time - start_time)
      print(total_time)

  print("Waiting for last task to be completed...")
  while prev_task.status()['state'] != 'COMPLETED' and prev_task.status()['state'] != 'FAILED':
      continue
  print("Last task completed!")

  total_time += (time.time() - end_time)
  print("Total Time Taken:", total_time)

Streaming output truncated to the last 5000 lines.
Task Started {'state': 'READY', 'description': 'Haora_15_2017', 'priority': 100, 'creation_timestamp_ms': 1762652761544, 'update_timestamp_ms': 1762652761544, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'JPA4UPIX3U5VCIKQ2SWCQECH', 'name': 'projects/ext-datasets/operations/JPA4UPIX3U5VCIKQ2SWCQECH'}
Grid 16
curr_year 2017
Saving data for Haora 2017
Task Started {'state': 'READY', 'description': 'Haora_16_2017', 'priority': 100, 'creation_timestamp_ms': 1762652766812, 'update_timestamp_ms': 1762652766812, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'EAMYVBQXNMMEY4FOMWM43BNC', 'name': 'projects/ext-datasets/operations/EAMYVBQXNMMEY4FOMWM43BNC'}
Grid 17
curr_year 2017
Saving data for Haora 2017
Task Started {'state': 'READY', 'description': 'Haora_17_2017', 'priority': 100, 'creation_timestamp_ms': 1762652773310, 'update_timestamp_ms': 1762652773310, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATUR